In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

train = pd.read_csv("trained_dataset.csv")
test = pd.read_csv("test_dataset.csv")

print(f"-> TRAINED DATASET SHAPE: {train.shape}")
print(f"-> TEST DATASET SHAPE: {test.shape}")

-> TRAINED DATASET SHAPE: (1460, 81)
-> TEST DATASET SHAPE: (1459, 80)


In [2]:
X_train = train.drop(columns='SalePrice')
y_train = train['SalePrice']
X_test = test.copy()

In [3]:
NullVals = X_train.isnull().sum()
NumricCols = X_train.select_dtypes(include=['int64', 'float64']).columns
MissingNumricCols = [i for i in NumricCols if NullVals[i] > 0]

CateCols = X_train.select_dtypes(include=['object']).columns
MissingCateCols = [i for i in CateCols if NullVals[i] > 0]

C:\Users\Admin\AppData\Local\Temp\ipykernel_7752\2752865170.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  CateCols = X_train.select_dtypes(include=['object']).columns


In [4]:
ColsForMean = ["LotFrontage"]
ColsForMedian = ["MasVnrArea", "GarageYrBlt"]
ColsForMode = ["Alley", "MasVnrType", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "Electrical", "FireplaceQu"]
ColsForMissing = ["GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC", "Fence", "MiscFeature"]

In [5]:
MeanColsImputer = Pipeline(steps=[("imputer", SimpleImputer(strategy="mean"))])
MedianColsImputer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
ModeColsImputer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
MisColsImputer = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="missing"))])

In [6]:
PreProcessor = ColumnTransformer(transformers=[
    ('mean_imputer', MeanColsImputer, ColsForMean),
    ('median_imputer', MedianColsImputer, ColsForMedian),
    ('mode_imputer', ModeColsImputer, ColsForMode),
    ('missing_imputer', MisColsImputer, ColsForMissing),
])

In [7]:
PreProcessor.fit(X_train)

MeanValues = PreProcessor.named_transformers_["mean_imputer"].named_steps["imputer"].statistics_
MedianValues = PreProcessor.named_transformers_["median_imputer"].named_steps["imputer"].statistics_
ModeValues = PreProcessor.named_transformers_["mode_imputer"].named_steps["imputer"].statistics_

In [8]:
X_trainClean = PreProcessor.transform(X_train)
print(X_trainClean)

[[65.0 196.0 2003.0 ... 'missing' 'missing' 'missing']
 [80.0 0.0 1976.0 ... 'missing' 'missing' 'missing']
 [68.0 162.0 2001.0 ... 'missing' 'missing' 'missing']
 ...
 [66.0 0.0 1941.0 ... 'missing' 'GdPrv' 'Shed']
 [68.0 0.0 1950.0 ... 'missing' 'missing' 'missing']
 [75.0 0.0 1965.0 ... 'missing' 'missing' 'missing']]


In [9]:
print(PreProcessor.transformers_)

X_trainCleanMiss = pd.DataFrame(X_trainClean, columns=ColsForMean + ColsForMedian + ColsForMode + ColsForMissing)

[('mean_imputer', Pipeline(steps=[('imputer', SimpleImputer())]), ['LotFrontage']), ('median_imputer', Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))]), ['MasVnrArea', 'GarageYrBlt']), ('mode_imputer', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent'))]), ['Alley', 'MasVnrType', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Electrical', 'FireplaceQu']), ('missing_imputer', Pipeline(steps=[('imputer',
                 SimpleImputer(fill_value='missing', strategy='constant'))]), ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature']), ('remainder', 'drop', ['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation

In [10]:
print(f"-> HERE IS TRANSFORM DATASET SHAPE: {X_trainCleanMiss.shape}\n")

MissCount = X_trainCleanMiss.isnull().sum().sum()
print(f"-> CHECK ANY MISSING VALUEs ARE NOT LEFT: {MissCount}")

-> HERE IS TRANSFORM DATASET SHAPE: (1460, 19)

-> CHECK ANY MISSING VALUEs ARE NOT LEFT: 0
